In [ ]:
import re
from pathlib import Path

import pandas as pd
from Bio import SeqIO

OUT = Path("results/sequences")
OUT.mkdir(parents=True, exist_ok=True)
SCREEN = Path("data/screening")
LEVEL = {"Complete Genome": 4, "Chromosome": 3, "Scaffold": 2, "Contig": 1}


def strain_tokens(text):
    cleaned = str(text).lower().replace("ena-yuan", " ")
    cleaned = re.sub(r"gc[af]_\d+\.\d+", " ", cleaned)
    cleaned = cleaned.replace("sp.", " ").replace("_", " ").replace(".", "-")
    cleaned = re.sub(r"[^a-z0-9 -]+", " ", cleaned)
    tokens = []
    for token in cleaned.split():
        if any(character.isdigit() for character in token) and len(token) >= 3 and token not in tokens:
            tokens.append(token)
    return tokens


def strain_group(organism, strain):
    genus = str(organism).split()[0].lower() if str(organism).strip() else "unknown"
    strain_text = str(strain).strip()
    if "ena-yuan" in strain_text.lower():
        tokens = strain_tokens(str(organism))
        return genus + "|" + "|".join(tokens) if tokens else str(organism).strip().lower()
    normalized = re.sub(r"[^a-z0-9]+", "-", strain_text.lower()).strip("-")
    return f"{genus}|{normalized}" if normalized else str(organism).strip().lower()


def qc_class(row):
    header = str(row.protein_accession)
    partial = "partial" in header.lower()
    passed = bool(row.passed_colap_screen) if pd.notna(row.passed_colap_screen) else False
    motifs = row.motif_matches if pd.notna(row.motif_matches) else None
    identity = row.identity_to_reference if pd.notna(row.identity_to_reference) else None
    coverage = row.coverage if pd.notna(row.coverage) else None
    length = row.protein_length
    gaps = row.mature_gap_columns if pd.notna(row.mature_gap_columns) else None
    if not passed:
        if identity is not None and identity >= 40 and (partial or (coverage is not None and coverage < 70) or length < 550):
            return "fragment", "coverage < 70% or length < 550"
        return "ambiguous_homolog", "below ColAP bitscore threshold"
    if motifs is not None and motifs < 6:
        return "ambiguous_homolog", "fewer than 6 functional residues"
    if partial or (gaps is not None and gaps > 20) or length < 550 or (coverage is not None and coverage < 70):
        return "fragment", "partial, gappy, or length < 550"
    if identity is not None and identity < 60:
        return "probable", "identity < 60%"
    return "presumed_ortholog", "passed ColAP filters"


genomes = pd.read_csv(SCREEN / "all_genomes.csv")
representatives = genomes[genomes["is_representative"]].copy()
candidates = pd.read_csv(SCREEN / "colap_candidates.csv")
presence = pd.read_csv(SCREEN / "colap_presence_absence.csv")
ogt_previous = pd.read_csv(SCREEN / "colap_ogt_metadata.csv")
hmm = {}
for line in (SCREEN / "pf01433.tbl").read_text(encoding="utf-8", errors="replace").splitlines():
    if not line or line.startswith("#"):
        continue
    parts = line.split()
    hmm[parts[0]] = float(parts[4])
sequences = {record.id: str(record.seq) for record in SeqIO.parse(SCREEN / "colap_candidates.faa", "fasta")}
candidates["pf01433_evalue"] = candidates["subject"].map(hmm)
candidates["pf01433_hit"] = candidates["pf01433_evalue"].notna()
classes = candidates.apply(qc_class, axis=1, result_type="expand")
candidates["qc_class"] = classes[0]
candidates["qc_reason"] = classes[1]
candidates["is_additional_copy"] = False
ranked = candidates[candidates["qc_class"].isin(["presumed_ortholog", "probable", "fragment"])].copy()
ranked = ranked.sort_values(["assembly_accession", "identity_to_reference", "colap_bitscore"], ascending=[True, False, False])
for _, group in ranked.groupby("assembly_accession"):
    if len(group) < 2:
        continue
    extras = group.index[1:]
    candidates.loc[extras, "is_additional_copy"] = True
    for index in extras:
        if candidates.at[index, "qc_class"] == "presumed_ortholog":
            candidates.at[index, "qc_class"] = "probable"
        if candidates.at[index, "qc_class"] == "probable":
            candidates.at[index, "qc_reason"] = "additional copy in the same genome"
meta = representatives.set_index("assembly_accession")
candidates["organism"] = candidates["assembly_accession"].map(meta["organism"])
candidates["strain"] = candidates["assembly_accession"].map(meta["strain"])
candidates["genus"] = candidates["assembly_accession"].map(meta["genus"])
candidates["strain_group"] = [strain_group(organism, strain) for organism, strain in zip(candidates["organism"].fillna(""), candidates["strain"].fillna(""))]
primary = candidates[candidates["qc_class"] == "presumed_ortholog"].copy()
primary["assembly_level"] = primary["assembly_accession"].map(meta["assembly_level"])
primary["checkm_completeness"] = primary["assembly_accession"].map(meta["checkm_completeness"])
primary["contig_count"] = primary["assembly_accession"].map(meta["contig_count"])
primary["level_rank"] = primary["assembly_level"].map(LEVEL).fillna(0)
primary["ena_mirror"] = primary["strain"].fillna("").str.contains("ena-yuan", case=False) | meta.loc[primary["assembly_accession"], "assembly_name"].fillna("").str.contains("ena-yuan", case=False).to_numpy()
primary = primary.sort_values(["strain_group", "ena_mirror", "level_rank", "contig_count", "checkm_completeness"], ascending=[True, True, False, True, False])
keep_index = []
for _, group in primary.groupby("strain_group", sort=False):
    keep_index.append(group.index[0])
    kept = sequences.get(candidates.at[group.index[0], "subject"], "")
    for index in group.index[1:]:
        other = sequences.get(candidates.at[index, "subject"], "")
        identical = bool(kept) and kept == other
        candidates.at[index, "qc_class"] = "repeat_assembly"
        candidates.at[index, "qc_reason"] = "repeat assembly, identical sequence" if identical else "repeat assembly, different sequence"
clean = candidates.loc[keep_index].copy()
if len(clean) != 202:
    raise SystemExit(f"expected 202 ColAP sequences, found {len(clean)}")
excluded = candidates[candidates["qc_class"] != "presumed_ortholog"][
    ["assembly_accession", "organism", "strain", "protein_accession", "protein_length", "identity_to_reference", "coverage", "qc_class", "qc_reason"]
]
excluded.to_csv(OUT / "excluded_sequences.csv", index=False)
with (OUT / "colap_clean.faa").open("w", encoding="utf-8") as output:
    missing = []
    for _, row in clean.iterrows():
        sequence = sequences.get(row.subject)
        if not sequence:
            missing.append(row.subject)
            continue
        output.write(f">{row.assembly_accession}|{row.protein_accession} {row.organism} strain={row.strain}\n")
        for start in range(0, len(sequence), 60):
            output.write(sequence[start:start + 60] + "\n")
if missing:
    raise SystemExit(f"missing sequences: {missing[:5]}")
ogt_by_assembly = ogt_previous.set_index("assembly_accession")
clean_meta = []
sequence_of = {}
for _, row in clean.iterrows():
    previous = ogt_by_assembly.loc[row.assembly_accession] if row.assembly_accession in ogt_by_assembly.index else None
    growth = bool(previous["has_measured_growth_curve"]) if previous is not None else False
    match = previous["ogt_match"] if previous is not None else "absent"
    ogt_value = previous["ratkowsky_ogt"] if previous is not None and match == "unique" else pd.NA
    if growth and match == "unique":
        ogt_type = "growth_curve"
        source = "Gentilhomme et al. 2024 growthrates.tsv"
    elif match == "unique":
        ogt_type = "ratkowsky_model"
        source = "Gentilhomme et al. 2024 Strain Ratk OGT Final.tsv"
    else:
        ogt_type = "absent"
        source = ""
    sequence_of[row.assembly_accession] = sequences[row.subject]
    clean_meta.append({
        "strain_group": row.strain_group,
        "organism": row.organism,
        "genus": row.genus,
        "strain": row.strain,
        "assembly_accession": row.assembly_accession,
        "protein_accession": row.protein_accession,
        "protein_length": row.protein_length,
        "identity_to_reference": row.identity_to_reference,
        "coverage": row.coverage,
        "motif_matches": row.motif_matches,
        "sequence_md5_length": len(sequences[row.subject]),
        "colap_present": True,
        "ogt_value": ogt_value,
        "ogt_type": ogt_type,
        "ogt_source": source,
        "paper_clade": "" if previous is None else previous["paper_clade"],
        "qc_class": "presumed_ortholog",
    })
clean_metadata = pd.DataFrame(clean_meta)
clean_metadata["sequence_identical_group"] = clean_metadata["assembly_accession"].map(
    lambda accession: min(other for other, sequence in sequence_of.items() if sequence == sequence_of[accession])
)
clean_metadata.to_csv(OUT / "colap_clean_metadata.csv", index=False)
status_rows = []
clean_accessions = set(clean["assembly_accession"])
assessable = set(presence.loc[presence.colap_status != "Not assessable", "assembly_accession"])
not_detected = set(presence.loc[presence.colap_status == "Not detected", "assembly_accession"])
for _, genome in representatives.iterrows():
    accession = genome.assembly_accession
    group = candidates[candidates["assembly_accession"] == accession]
    if accession in clean_accessions:
        status = "presumed_ortholog"
        reason = "clean set"
    elif accession not in assessable:
        status = "not_assessable"
        reason = "no annotated proteome"
    elif (group["qc_class"] == "repeat_assembly").any():
        status = "repeat_assembly"
        reason = group.loc[group.qc_class == "repeat_assembly", "qc_reason"].iloc[0]
    elif (group["qc_class"] == "probable").any() and not (group["qc_class"] == "presumed_ortholog").any():
        status = "probable"
        reason = "identity < 60%"
    elif (group["qc_class"] == "fragment").any() and not (group["qc_class"] == "presumed_ortholog").any():
        status = "fragment"
        reason = "fragment only"
    elif len(group) and (group["qc_class"] == "ambiguous_homolog").any() and not (group["qc_class"].isin(["presumed_ortholog", "probable", "fragment"])).any():
        status = "ambiguous_homolog"
        reason = "M1 homologs below ColAP filters"
    elif accession in not_detected:
        status = "not_detected"
        reason = "no hit at E-value 1e-5"
    else:
        status = "ambiguous_homolog"
        reason = "no complete ColAP ortholog"
    status_rows.append({
        "organism": genome.organism,
        "genus": genome.genus,
        "strain": genome.strain,
        "strain_group": strain_group(genome.organism, genome.strain),
        "assembly_accession": accession,
        "biosample_accession": genome.biosample_accession,
        "assembly_level": genome.assembly_level,
        "source_database": genome.source_database,
        "qc_status": status,
        "reason": reason,
    })
pd.DataFrame(status_rows).to_csv(OUT / "genome_status.csv", index=False)
